In [28]:
import pandas as pd
import numpy as np

sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Material_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Inlet_Draft"
]

df = pd.read_csv("../data/Data.csv")

df.head()

,time,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft
0,01-01-2017 00:00,867.63,910.42,-189.54,-186.04,852.13,-145.9
1,01-01-2017 00:05,879.23,918.14,-184.33,-182.1,862.53,-149.76
2,01-01-2017 00:10,875.67,924.18,-181.26,-166.47,866.06,-145.01
3,01-01-2017 00:15,875.28,923.15,-179.15,-174.83,865.85,-142.82
4,01-01-2017 00:20,891.66,934.26,-178.32,-173.72,876.06,-143.39


In [29]:
# Convert sensor columns to numeric
for col in sensor_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Parse timestamp
df["time_dt"] = pd.to_datetime(
    df["time"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

# Sort chronologically
df = df.sort_values("time_dt").reset_index(drop=True)

# Keep only rows where all six sensors have numeric values
ts_df = df[["time_dt"] + sensor_cols].dropna(
    subset=sensor_cols
).copy()

print("Original shape:", df.shape)
print("Modeling shape:", ts_df.shape)
print("Missing sensor values:", ts_df[sensor_cols].isna().sum().sum())

Original shape: (377719, 8)
Modeling shape: (376124, 7)
Missing sensor values: 0


In [30]:
# Check time intervals between consecutive observations
time_diff = ts_df["time_dt"].diff()

print(time_diff.value_counts().head(10))

time_dt
0 days 00:05:00    375767
0 days 00:10:00       303
0 days 00:15:00         7
0 days 00:30:00         5
0 days 01:05:00         5
0 days 00:25:00         4
0 days 02:00:00         4
0 days 00:45:00         3
0 days 01:00:00         3
1 days 00:00:00         3
Name: count, dtype: int64


In [31]:
# Show gaps larger than the expected 5-minute interval
gaps = ts_df.loc[
    time_diff > pd.Timedelta(minutes=5),
    ["time_dt"]
].copy()

gaps["previous_time"] = ts_df["time_dt"].shift(1).loc[gaps.index]
gaps["gap"] = gaps["time_dt"] - gaps["previous_time"]

gaps

,time_dt,previous_time,gap
2492,2017-01-09 15:40:00,2017-01-09 13:50:00,0 days 01:50:00
2899,2017-01-11 01:35:00,2017-01-10 23:00:00,0 days 02:35:00
4570,2017-01-16 20:50:00,2017-01-16 20:40:00,0 days 00:10:00
5598,2017-01-20 10:30:00,2017-01-20 10:20:00,0 days 00:10:00
7304,2017-01-26 08:40:00,2017-01-26 08:30:00,0 days 00:10:00
...,...,...,...
375586,2020-07-31 02:35:00,2020-07-31 02:25:00,0 days 00:10:00
375851,2020-08-01 00:40:00,2020-08-01 00:30:00,0 days 00:10:00
376011,2020-08-01 14:00:00,2020-08-01 13:50:00,0 days 00:10:00
376886,2020-08-04 14:55:00,2020-08-04 14:45:00,0 days 00:10:00


In [32]:
# Identify continuous 5-minute segments
ts_df["time_diff"] = ts_df["time_dt"].diff()

ts_df["segment_id"] = (
    ts_df["time_diff"] != pd.Timedelta(minutes=5)
).cumsum()

segment_sizes = (
    ts_df.groupby("segment_id")
    .size()
    .sort_values(ascending=False)
)

print("Number of continuous segments:", len(segment_sizes))
print("\nLargest segments:")
print(segment_sizes.head(10))

Number of continuous segments: 357

Largest segments:
segment_id
6      8072
268    7300
117    7199
319    7066
217    6602
169    6216
307    6011
326    5978
221    5741
213    5605
dtype: int64


In [33]:
from statsmodels.tsa.api import VAR

# Select the largest continuous training segment
largest_segment_id = segment_sizes.index[0]

var_data = (
    ts_df[ts_df["segment_id"] == largest_segment_id]
    [sensor_cols]
    .copy()
)

print("Selected segment:", largest_segment_id)
print("Rows:", len(var_data))


Selected segment: 6
Rows: 8072


In [34]:
from statsmodels.tsa.api import VAR
from sklearn.preprocessing import StandardScaler

# Scale the continuous segment
var_scaler = StandardScaler()
var_scaled = var_scaler.fit_transform(var_data)

var_scaled = pd.DataFrame(
    var_scaled,
    columns=sensor_cols
)

# Create VAR model
var_model = VAR(var_scaled)

# Test lags up to 12 (1 hour of history)
lag_selection = var_model.select_order(maxlags=12)

print(lag_selection.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -5.076      -5.071    0.006243      -5.074
1       -11.97      -11.94   6.314e-06      -11.96
2       -13.41      -13.34   1.500e-06      -13.39
3       -13.66      -13.56   1.165e-06      -13.63
4       -13.77     -13.64*   1.047e-06      -13.72
5       -13.80      -13.64   1.015e-06      -13.75
6       -13.82      -13.63   9.916e-07      -13.76
7       -13.84      -13.62   9.763e-07      -13.76
8       -13.85      -13.60   9.623e-07     -13.77*
9       -13.85      -13.57   9.617e-07      -13.76
10      -13.86      -13.54   9.602e-07      -13.75
11      -13.86      -13.51   9.590e-07      -13.74
12     -13.86*      -13.48  9.549e-07*      -13.73
--------------------------------------------------


In [35]:
# Fit VAR with 4 lags (20 minutes)
var_model_4 = VAR(var_scaled)

var_result = var_model_4.fit(4)

print(var_result.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 25, Sep, 2026
Time:                     16:29:30
--------------------------------------------------------------------
No. of Equations:         6.00000    BIC:                   -13.5417
Nobs:                     8068.00    HQIC:                  -13.6273
Log likelihood:          -13386.1    FPE:                1.15459e-06
AIC:                     -13.6718    Det(Omega_mle):     1.13336e-06
--------------------------------------------------------------------
Results for equation Cyclone_Inlet_Gas_Temp
                                 coefficient       std. error           t-stat            prob
----------------------------------------------------------------------------------------------
const                              -0.000268         0.009171           -0.029           0.977
L1.Cyclone_Inlet_Gas_Temp           0.392993         0.027203           14.447    

In [36]:
# Create lag features within each continuous segment

lagged_df = ts_df.copy()

for lag in [1, 2, 3]:
    for col in sensor_cols:
        lagged_df[f"{col}_lag{lag}"] = (
            lagged_df.groupby("segment_id")[col].shift(lag)
        )

# Remove rows where the required lag history is unavailable
lagged_df = lagged_df.dropna().reset_index(drop=True)

print("Lagged dataset shape:", lagged_df.shape)

Lagged dataset shape: (375070, 27)


In [37]:
# Current sensor values = targets
y_cols = sensor_cols

# Lagged sensor values = predictors
X_cols = [
    f"{col}_lag{lag}"
    for lag in [1, 2, 3]
    for col in sensor_cols
]

X = lagged_df[X_cols].copy()
y = lagged_df[y_cols].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (375070, 18)
y shape: (375070, 6)


In [38]:
# Chronological train-test split

split_idx = int(len(lagged_df) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()

y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(lagged_df["time_dt"].iloc[:split_idx].min())
print("to")
print(lagged_df["time_dt"].iloc[:split_idx].max())

print("\nTesting period:")
print(lagged_df["time_dt"].iloc[split_idx:].min())
print("to")
print(lagged_df["time_dt"].iloc[split_idx:].max())

Training samples: 300056
Testing samples: 75014

Training period:
2017-01-01 00:15:00
to
2019-11-19 02:10:00

Testing period:
2019-11-19 02:15:00
to
2020-08-07 12:15:00


In [39]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:", y_train_scaled.shape)
print("y_test:", y_test_scaled.shape)

X_train: (300056, 18)
X_test: (75014, 18)
y_train: (300056, 6)
y_test: (75014, 6)


In [40]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(
    X_train_scaled,
    y_train_scaled
)

print("Model trained successfully.")

Model trained successfully.


In [41]:
from sklearn.linear_model import Ridge

ridge_var = Ridge(alpha=1.0)

ridge_var.fit(
    X_train_scaled,
    y_train_scaled
)

print("Ridge VAR model trained successfully.")

Ridge VAR model trained successfully.


In [42]:
y_pred_scaled = ridge_var.predict(X_test_scaled)

print("Prediction shape:", y_pred_scaled.shape)

Prediction shape: (75014, 6)


In [43]:
y_pred = scaler_y.inverse_transform(y_pred_scaled)

y_pred = pd.DataFrame(
    y_pred,
    columns=sensor_cols,
    index=y_test.index
)

y_pred.head()

,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft
300056,925.595903,965.426283,-275.333074,-236.969681,918.885337,-220.325698
300057,914.958288,958.082348,-280.914521,-241.214682,911.900736,-224.664361
300058,896.892842,944.000412,-280.941388,-239.877829,904.537192,-226.953774
300059,895.232223,939.204031,-285.243067,-237.240307,899.455853,-230.709850
300060,880.208280,908.466252,-289.794918,-240.959516,883.243746,-234.966492


In [44]:
# Residuals in standardized space
residual_scaled = y_test_scaled - y_pred_scaled

print("Residual shape:", residual_scaled.shape)

Residual shape: (75014, 6)


In [45]:
lagged_df = ts_df.copy()

for lag in [1, 2, 3, 4]:
    for col in sensor_cols:
        lagged_df[f"{col}_lag{lag}"] = (
            lagged_df.groupby("segment_id")[col].shift(lag)
        )

lagged_df = lagged_df.dropna().reset_index(drop=True)

print("Lagged dataset shape:", lagged_df.shape)

Lagged dataset shape: (374726, 33)


In [46]:
y_cols = sensor_cols

X_cols = [
    f"{col}_lag{lag}"
    for lag in [1, 2, 3, 4]
    for col in sensor_cols
]

X = lagged_df[X_cols].copy()
y = lagged_df[y_cols].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (374726, 24)
y shape: (374726, 6)


In [47]:
split_idx = int(len(lagged_df) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()

y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(lagged_df["time_dt"].iloc[:split_idx].min())
print("to")
print(lagged_df["time_dt"].iloc[:split_idx].max())

print("\nTesting period:")
print(lagged_df["time_dt"].iloc[split_idx:].min())
print("to")
print(lagged_df["time_dt"].iloc[split_idx:].max())

Training samples: 299780
Testing samples: 74946

Training period:
2017-01-01 00:20:00
to
2019-11-19 02:45:00

Testing period:
2019-11-19 02:50:00
to
2020-08-07 12:15:00


In [48]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:", y_train_scaled.shape)
print("y_test:", y_test_scaled.shape)

X_train: (299780, 24)
X_test: (74946, 24)
y_train: (299780, 6)
y_test: (74946, 6)


In [49]:
from sklearn.linear_model import Ridge

ridge_var = Ridge(alpha=1.0)

ridge_var.fit(
    X_train_scaled,
    y_train_scaled
)

print("Ridge multivariate time-series model trained successfully.")

Ridge multivariate time-series model trained successfully.


In [50]:
y_pred_scaled = ridge_var.predict(X_test_scaled)

print("Prediction shape:", y_pred_scaled.shape)

Prediction shape: (74946, 6)


In [51]:
y_pred = scaler_y.inverse_transform(y_pred_scaled)

y_pred = pd.DataFrame(
    y_pred,
    columns=sensor_cols,
    index=y_test.index
)

print("Prediction shape:", y_pred.shape)
y_pred.head()

Prediction shape: (74946, 6)


,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft
299780,885.041888,913.776516,-279.406475,-233.090547,874.567609,-226.331839
299781,889.910930,927.396771,-284.102497,-239.395065,880.374878,-229.814388
299782,885.433048,924.357416,-291.064874,-248.447477,879.041991,-235.056808
299783,915.071573,954.805677,-291.065409,-248.734980,895.558203,-235.436039
299784,905.346504,950.291969,-288.894148,-245.627970,897.990597,-233.874219


In [52]:
residual_scaled = y_test_scaled - y_pred_scaled

print("Residual shape:", residual_scaled.shape)

Residual shape: (74946, 6)


In [53]:
residual_scaled = y_test_scaled - y_pred_scaled

residuals = pd.DataFrame(
    residual_scaled,
    columns=sensor_cols,
    index=y_test.index
)

residuals.head()

,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft
299780,0.029949,0.079907,-0.071005,-0.108686,0.029800,-0.053493
299781,-0.032612,-0.020602,-0.101235,-0.160507,0.007120,-0.116123
299782,0.188063,0.157753,-0.041258,-0.049142,0.065018,-0.064756
299783,-0.052079,-0.019925,0.032681,0.074244,0.031142,0.010185
299784,-0.149796,-0.228280,0.029278,0.102918,-0.055779,0.031049


### ANOMALY SCORING & LEAKAGE-SAFE THRESHOLDING

To identify abnormal process periods without data leakage:
1. **Training Residual Distribution**: Compute the multivariate prediction residual vector in standardized space for training samples: $e_{\text{train}} = y_{\text{train, scaled}} - \hat{y}_{\text{train, scaled}}$.
2. **Anomaly Score**: Calculate the Euclidean norm (magnitude) of the standardized residual vector across all 6 sensors: $S = \|e\|_2 = \sqrt{\sum_{i=1}^6 e_i^2}$.
3. **Leakage-Safe Threshold**: Determine the 99th percentile threshold strictly from training anomaly scores ($S_{\text{train}}$).
4. **Test Set Evaluation & Episode Consolidation**: Apply the training threshold to test anomaly scores ($S_{\text{test}}$) and group consecutive flagged 5-minute timestamps into contiguous abnormal episodes.

In [54]:
# Calculate training predictions and standardized residuals
y_train_pred_scaled = ridge_var.predict(X_train_scaled)
train_residuals_scaled = y_train_scaled - y_train_pred_scaled

# Compute multivariate anomaly scores (L2 norm of standardized residuals) on training set
train_anomaly_scores = np.linalg.norm(train_residuals_scaled, axis=1)

# Set leakage-safe anomaly threshold at 99th percentile of training residuals
anomaly_threshold = np.percentile(train_anomaly_scores, 99.0)

print("Training Residual Distribution Summary:")
print(f"Mean residual score: {np.mean(train_anomaly_scores):.4f}")
print(f"Std residual score:  {np.std(train_anomaly_scores):.4f}")
print(f"Max residual score:  {np.max(train_anomaly_scores):.4f}")
print(f"\nLeakage-safe 99th percentile threshold (from training set): {anomaly_threshold:.4f}")


Training Residual Distribution Summary:
Mean residual score: 0.1608
Std residual score:  0.1963
Max residual score:  8.4741

Leakage-safe 99th percentile threshold (from training set): 0.8786


In [57]:
# Compute test set multivariate anomaly scores
test_anomaly_scores = np.linalg.norm(residual_scaled, axis=1)

# Extract test timestamps and flags
test_df = lagged_df.iloc[split_idx:].copy().reset_index(drop=True)
test_df["anomaly_score"] = test_anomaly_scores
test_df["is_anomaly"] = test_anomaly_scores >= anomaly_threshold

flagged_points = test_df[test_df["is_anomaly"]].copy().reset_index(drop=True)

print(f"Test samples: {len(test_df)}")
print(f"Flagged anomaly timestamps: {len(flagged_points)} ({len(flagged_points)/len(test_df):.2%})")

# Group consecutive 5-minute flagged timestamps into abnormal episodes
flagged_points["time_gap"] = flagged_points["time_dt"].diff()
flagged_points["episode_id"] = (flagged_points["time_gap"] > pd.Timedelta(minutes=5)).cumsum()

# Summarize episodes
episodes = []
for ep_id, group in flagged_points.groupby("episode_id"):
    start_t = group["time_dt"].min()
    end_t = group["time_dt"].max()
    dur_min = int((end_t - start_t).total_seconds() / 60) + 5
    count = len(group)
    max_score = group["anomaly_score"].max()
    mean_score = group["anomaly_score"].mean()
    
    # Identify primary driving sensor (highest absolute standardized residual during episode)
    ep_test_indices = group.index
    ep_res_df = pd.DataFrame(residual_scaled[ep_test_indices], columns=sensor_cols)
    primary_sensor = ep_res_df.abs().mean().idxmax()
    
    episodes.append({
        "episode_id": ep_id + 1,
        "start_time": start_t,
        "end_time": end_t,
        "duration_minutes": dur_min,
        "flagged_points": count,
        "max_anomaly_score": round(max_score, 4),
        "mean_anomaly_score": round(mean_score, 4),
        "primary_sensor_driver": primary_sensor
    })

episodes_df = pd.DataFrame(episodes)

print(f"\nExtracted {len(episodes_df)} abnormal episodes from testing set.")
print(episodes_df.head(15))


Test samples: 74946
Flagged anomaly timestamps: 379 (0.51%)

Extracted 203 abnormal episodes from testing set.
    episode_id          start_time            end_time  duration_minutes  \
0            1 2019-11-20 05:10:00 2019-11-20 05:15:00                10   
1            2 2019-11-20 05:50:00 2019-11-20 05:50:00                 5   
2            3 2019-11-20 16:10:00 2019-11-20 16:15:00                10   
3            4 2019-11-20 17:10:00 2019-11-20 17:10:00                 5   
4            5 2019-11-20 17:50:00 2019-11-20 17:50:00                 5   
5            6 2019-11-22 00:35:00 2019-11-22 00:40:00                10   
6            7 2019-11-28 13:35:00 2019-11-28 13:35:00                 5   
7            8 2019-11-29 11:40:00 2019-11-29 11:45:00                10   
8            9 2019-12-06 18:40:00 2019-12-06 18:45:00                10   
9           10 2019-12-07 12:05:00 2019-12-07 12:15:00                15   
10          11 2019-12-14 03:00:00 2019-12-14 03:05:0

In [58]:
import os

output_dir = "../outputs/results"
os.makedirs(output_dir, exist_ok=True)

# Save episode summary CSV
episodes_path = os.path.join(output_dir, "multivariate_ridge_abnormal_episodes.csv")
episodes_df.to_csv(episodes_path, index=False)

# Save test point anomaly scores CSV
scores_path = os.path.join(output_dir, "multivariate_ridge_test_scores.csv")
test_scores_df = test_df[["time_dt", "anomaly_score", "is_anomaly"] + sensor_cols]
test_scores_df.to_csv(scores_path, index=False)

print(f"Saved abnormal episodes summary to: {episodes_path}")
print(f"Saved test point anomaly scores to:  {scores_path}")


Saved abnormal episodes summary to: ../outputs/results\multivariate_ridge_abnormal_episodes.csv
Saved test point anomaly scores to:  ../outputs/results\multivariate_ridge_test_scores.csv
